In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)

In [2]:
import os
!git clone https://github.com/bencejdanko/bert-tweeteval

# ensure latest
os.chdir('/content/bert-tweeteval')
!cd /content/bert-tweeteval && git pull

Cloning into 'bert-tweeteval'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 200 (delta 126), reused 112 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (200/200), 769.02 KiB | 8.64 MiB/s, done.
Resolving deltas: 100% (126/126), done.
Already up to date.


In [3]:
# copy over package
PACKAGE = "src"

import sys
sys.path.append(f"/content/bert-tweeteval/{PACKAGE}")

In [4]:
from download import download_and_split_dataset
from train import train_and_evaluate

import pandas as pd
from datasets import Dataset

from corruption import create_corruption_ablations
from domain_shift import create_shift_ablation_sets
from transformers import Trainer, AutoModelForSequenceClassification, AutoTokenizer


In [5]:
id_labels = {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
labels_id = {"anger": 0, "joy": 1, "optimism": 2, "sadness": 3}
candidate_labels = list(id_labels.values())
hypothesis_template = "This tweet expresses {}."

In [6]:
train_df, val_df, test_df = download_and_split_dataset()
print(f"Training set: {len(train_df)}")
print(f"Validation set: {len(val_df)}")
print(f"Testing set: {len(test_df)}")

README.md: 0.00B [00:00, ?B/s]

emotion/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

emotion/test-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

emotion/validation-00000-of-00001.parque(…):   0%|          | 0.00/28.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

Training set: 3257
Validation set: 374
Testing set: 1421


In [7]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [8]:
ft_results = {}

In [10]:
import torch
import os
import sys

In [9]:
from train import evaluate # Updated to a simplified eval if needed or from src
import pandas as pd
from transformers import Trainer, AutoModelForSequenceClassification, AutoTokenizer
from corruption import create_corruption_ablations
from domain_shift import create_shift_ablation_sets

In [11]:
repo_id = "bdanko/bert-tweeteval"
model_names = [f"{repo_id}-distilbert", f"{repo_id}-distilroberta"]
ft_results = {}

for model_name in model_names:
    print(f"Evaluating model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    trainer = Trainer(model=model)

    ft_results[model_name] = {}

    # Baseline evaluation
    ft_results[model_name]["standard"] = evaluate(trainer, tokenizer, test_df, model_name, candidate_labels)

    # Corruption Ablations
    print(f"\nRunning corruption ablations for {model_name}")
    corrupted_dfs = create_corruption_ablations(test_df)
    ft_results[model_name]["corruptions"] = {}
    for name, c_df in corrupted_dfs.items():
        ft_results[model_name]["corruptions"][name] = evaluate(trainer, tokenizer, c_df, f"{model_name}_{name}", candidate_labels)

    # Domain Shift Ablations
    print(f"\nRunning domain shift ablations {model_name}")
    shift_dfs = create_shift_ablation_sets(test_df)
    ft_results[model_name]["shifts"] = {}
    for name, s_df in shift_dfs.items():
        ft_results[model_name]["shifts"][name] = evaluate(trainer, tokenizer, s_df, f"{model_name}_{name}", candidate_labels)


Evaluating model: bdanko/bert-tweeteval-distilbert


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Evaluating bdanko/bert-tweeteval-distilroberta_no_links on provided dataset (size: 1421)...


Map:   0%|          | 0/673 [00:00<?, ? examples/s]

Evaluating bdanko/bert-tweeteval-distilroberta_with_hashtags on provided dataset (size: 673)...


Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Evaluating bdanko/bert-tweeteval-distilroberta_no_hashtags on provided dataset (size: 748)...


In [19]:
import pandas as pd

all_rows = []

for model_path, categories in ft_results.items():
    model_name = model_path.split('/')[-1]

    for cat_type in ["corruptions", "shifts"]:
        for test_name, res in categories.get(cat_type, {}).items():
            macro_avg = res['Classification Report Dict']['macro avg']
            all_rows.append({
                'Type': cat_type.capitalize(),
                'Test': test_name,
                'Model': model_name,
                'Accuracy': res['Accuracy'],
                'Macro F1': res['Macro F1'],
                'Macro Precision': macro_avg['precision'],
                'Macro Recall': macro_avg['recall'],
                'ECE': res['ECE']
            })

df = pd.DataFrame(all_rows)

unified_df = df.pivot_table(
    index=['Type', 'Test'],
    columns='Model',
    values=['Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall', 'ECE']
)

unified_df = unified_df.swaplevel(0, 1, axis=1).sort_index(axis=1)

display(unified_df)


Model                           bert-tweeteval-distilbert                      \
                                                 Accuracy       ECE  Macro F1   
Type        Test                                                                
Corruptions corruption_all                       0.777621  0.186762  0.731208   
            corruption_emojis                    0.796622  0.169403  0.759471   
            corruption_hashtags                  0.797326  0.169666  0.758076   
            corruption_typos                     0.774806  0.192172  0.735679   
            original                             0.798030  0.169843  0.761196   
Shifts      full_test                            0.798030  0.169843  0.761196   
            no_hashtags                          0.779412  0.183594  0.729214   
            no_links                             0.798030  0.169843  0.761196   
            no_mentions                          0.786865  0.178595  0.764952   
            with_hashtags                        0.818722  0.155966  0.791053   
            with_mentions                        0.812704  0.158342  0.724186   

Model                                                         \
                                Macro Precision Macro Recall   
Type        Test                                               
Corruptions corruption_all             0.748215     0.719606   
            corruption_emojis          0.765310     0.754525   
            corruption_hashtags        0.767174     0.751265   
            corruption_typos           0.750336     0.726397   
            original                   0.767879     0.756296   
Shifts      full_test                  0.767879     0.756296   
            no_hashtags                0.740608     0.721849   
            no_links                   0.767879     0.756296   
            no_mentions                0.763810     0.766616   
            with_hashtags              0.795572     0.789115   
            with_mentions              0.761235     0.705813   

Model                           bert-tweeteval-distilroberta            \
                                                    Accuracy       ECE   
Type        Test                                                         
Corruptions corruption_all                          0.769880  0.027935   
            corruption_emojis                       0.788177  0.044574   
            corruption_hashtags                     0.790289  0.049607   
            corruption_typos                        0.762139  0.032455   
            original                                0.788881  0.042115   
Shifts      full_test                               0.788881  0.042115   
            no_hashtags                             0.783422  0.055099   
            no_links                                0.788881  0.042115   
            no_mentions                             0.788104  0.041246   
            with_hashtags                           0.794948  0.038689   
            with_mentions                           0.789902  0.046546   

Model                                                                   
                                 Macro F1 Macro Precision Macro Recall  
Type        Test                                                        
Corruptions corruption_all       0.729321        0.785784     0.704836  
            corruption_emojis    0.754264        0.799723     0.730673  
            corruption_hashtags  0.750437        0.804039     0.727826  
            corruption_typos     0.719843        0.767177     0.700259  
            original             0.750644        0.799548     0.728545  
Shifts      full_test            0.750644        0.799548     0.728545  
            no_hashtags          0.730822        0.795115     0.706223  
            no_links             0.750644        0.799548     0.728545  
            no_mentions          0.763955        0.796607     0.749939  
            with_hashtags        0.767394        0.8042

In [17]:
unified_df.to_markdown()

"|                                        |   ('bert-tweeteval-distilbert', 'Accuracy') |   ('bert-tweeteval-distilbert', 'ECE') |   ('bert-tweeteval-distilbert', 'Macro F1') |   ('bert-tweeteval-distilbert', 'Macro Precision') |   ('bert-tweeteval-distilbert', 'Macro Recall') |   ('bert-tweeteval-distilroberta', 'Accuracy') |   ('bert-tweeteval-distilroberta', 'ECE') |   ('bert-tweeteval-distilroberta', 'Macro F1') |   ('bert-tweeteval-distilroberta', 'Macro Precision') |   ('bert-tweeteval-distilroberta', 'Macro Recall') |\n|:---------------------------------------|--------------------------------------------:|---------------------------------------:|--------------------------------------------:|---------------------------------------------------:|------------------------------------------------:|-----------------------------------------------:|------------------------------------------:|-----------------------------------------------:|------------------------------------------------